In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

In [2]:
DATA_PATH = Path("/home/ldominguez/master_ml/TFM_Store_Sales/data/raw")

train = pd.read_csv(DATA_PATH / "train.csv")
test = pd.read_csv(DATA_PATH / "test.csv")
stores = pd.read_csv(DATA_PATH / "stores.csv")
transactions = pd.read_csv(DATA_PATH / "transactions.csv")
oil = pd.read_csv(DATA_PATH / "oil.csv")
holidays = pd.read_csv(DATA_PATH / "holidays_events.csv")
sample_submission = pd.read_csv(DATA_PATH / "sample_submission.csv")

In [3]:
def describe_dataframe(df, name):
    print("=" * 80)
    print(name)
    print("=" * 80)

    print(f"Filas: {df.shape[0]:,}")
    print(f"Columnas: {df.shape[1]}")

    print("\nTipos de datos")
    print(df.dtypes)

    print("\nValores nulos")
    print(df.isnull().sum())

    print("\nDuplicados")
    print(df.duplicated().sum())

    print("\nPrimeras filas")
    display(df.head())

In [4]:
describe_dataframe(train, "TRAIN")

TRAIN
Filas: 3,000,888
Columnas: 6

Tipos de datos
id               int64
date            object
store_nbr        int64
family          object
sales          float64
onpromotion      int64
dtype: object

Valores nulos
id             0
date           0
store_nbr      0
family         0
sales          0
onpromotion    0
dtype: int64

Duplicados
0

Primeras filas


,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


# TRAIN

1. ¿Cuántas filas tiene?
    - Hay 3.000.888 filas y 6 columnas

2. ¿Qué representa una fila?
    - Cada fila representa las ventas diarias de una familia de productos en una tienda concreta en una fecha determinada.

3. ¿Qué variable queremos predecir?
    - La variable objetivo es *sales*, que representa el volumen de ventas diarias para cada combinación de fecha, tienda y familia de productos.

    - El objetivo principal del TFM será construir modelos capaces de estimar esta variable utilizando la información histórica y las variables auxiliares disponibles.

4. ¿Qué columnas son categóricas?
    
    Las variables categóricas son:
        - *store_nbr* (identificador de tienda; aunque aparece como entero, representa una categoría y no una magnitud numérica).
        - *family* (familia de productos).

    Además:
        - *date* es una variable temporal (actualmente cargada como object, aunque posteriormente se convertirá al tipo datetime).
        - *id* es un identificador único de cada observación y no se utilizará como variable predictora.
        - *sales* es una variable numérica continua.
        - *onpromotion* es una variable numérica discreta que indica el número de productos en promoción.

5. ¿Hay nulos?
    - No. El conjunto de entrenamiento no presenta valores nulos en ninguna de sus variables, por lo que no será necesario realizar tratamientos específicos de imputación en esta tabla.

6. ¿Hay duplicados?
    - No. No se han encontrado registros duplicados en el conjunto de entrenamiento, por lo que cada observación corresponde a una combinación única de fecha, tienda y familia de productos.

Aunque *id* aparece en train.csv, no contiene información útil para el modelo, ya que es únicamente un identificador único de cada registro. Por tanto, en las fases posteriores de modelado no se utilizará como variable predictora.

In [5]:
describe_dataframe(test, "TEST")

TEST
Filas: 28,512
Columnas: 5

Tipos de datos
id              int64
date           object
store_nbr       int64
family         object
onpromotion     int64
dtype: object

Valores nulos
id             0
date           0
store_nbr      0
family         0
onpromotion    0
dtype: int64

Duplicados
0

Primeras filas


,id,date,store_nbr,family,onpromotion
0,3000888,2017-08-16,1,AUTOMOTIVE,0
1,3000889,2017-08-16,1,BABY CARE,0
2,3000890,2017-08-16,1,BEAUTY,2
3,3000891,2017-08-16,1,BEVERAGES,20
4,3000892,2017-08-16,1,BOOKS,0


# TEST 

1. ¿Qué diferencia tiene respecto a TRAIN?

El archivo *test.csv* contiene la misma estructura básica que *train.csv*, pero **no incluye la variable objetivo sales**, ya que su finalidad es **evaluar la capacidad predictiva de los modelos desarrollados**.

En concreto:

Contiene *28.512 observaciones* correspondientes a **fechas posteriores al conjunto de entrenamiento**.
Incluye las variables:
    - *id*
    - *date*
    - *store_nbr*
    - *family*
    - *onpromotion*

**No contiene** la columna *sales*, que es precisamente la **variable que deberá ser estimada por el modelo**.

Una vez entrenado el modelo utilizando train.csv, las predicciones obtenidas para sales sobre las observaciones de test.csv deberán almacenarse en el formato indicado por sample_submission.csv para su evaluación. 

**Observación importante** (muy recomendable incluirla en el TFM)

El conjunto train.csv abarca el periodo comprendido entre el 1 de enero de 2013 y el 15 de agosto de 2017, mientras que test.csv corresponde a los 16 días siguientes, desde el 16 de agosto de 2017 hasta el 31 de agosto de 2017.

Esto confirma que se trata de un problema de predicción de series temporales, donde el modelo debe aprender a partir de datos históricos para estimar ventas en fechas futuras.

Este último punto es especialmente importante: no basta con decir que a test "le falta la columna sales", sino explicar por qué. El conjunto de prueba representa un periodo futuro, lo que obliga a entrenar y validar los modelos respetando el orden cronológico de los datos.

De hecho, este detalle justifica una decisión metodológica que se tomará más adelante: no podrá usarse una división aleatoria (`train_test_split` con `shuffle=True`), sino una validación temporal.

In [6]:
describe_dataframe(stores, "STORES")

STORES
Filas: 54
Columnas: 5

Tipos de datos
store_nbr     int64
city         object
state        object
type         object
cluster       int64
dtype: object

Valores nulos
store_nbr    0
city         0
state        0
type         0
cluster      0
dtype: int64

Duplicados
0

Primeras filas


,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8
3,4,Quito,Pichincha,D,9
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4


# STORES

1. ¿Qué información aporta?

El archivo *stores.csv* contiene **información descriptiva de cada una de las 54 tiendas incluidas en el dataset**.

Cada *registro* corresponde a una *tienda* e incluye las siguientes características:

- *store_nbr*: identificador único de la tienda.
- *city*: ciudad donde se encuentra la tienda.
- *state*: provincia o estado al que pertenece la tienda.
- *type*: tipo de establecimiento (A, B, C, D o E).
- *cluster*: grupo o clúster al que pertenece la tienda según la clasificación realizada en el dataset.

Estas variables **aportan información contextual** sobre cada establecimiento y podrán utilizarse como variables predictoras durante la fase de modelado, ya que **permiten capturar diferencias entre tiendas derivadas de su ubicación o características**.

2. ¿Qué columna sirve para unirla con TRAIN?

La unión entre *stores.csv* y *train.csv* se realiza mediante la *columna store_nbr*, que actúa como **identificador único de cada tienda**.

De esta forma, a cada registro de ventas del conjunto de entrenamiento se le pueden añadir las características correspondientes de la tienda donde se produjeron dichas ventas.

La **relación entre ambos conjuntos de datos** es de tipo *uno a muchos (1:N)*:

- **Una tienda aparece una sola vez** en *stores.csv*.
- **Esa misma tienda puede aparecer miles de veces en train.csv**, ya que existen registros de ventas para múltiples fechas y familias de productos.

La información contenida en stores.csv será incorporada al conjunto de entrenamiento durante la fase de preparación de datos mediante una operación de unión (merge) utilizando la variable store_nbr. Esto permitirá que los modelos predictivos tengan en cuenta características propias de cada tienda, como su ubicación geográfica, tipo o clúster, que podrían influir en el comportamiento de las ventas.

In [7]:
describe_dataframe(transactions, "TRANSACTIONS")

TRANSACTIONS
Filas: 83,488
Columnas: 3

Tipos de datos
date            object
store_nbr        int64
transactions     int64
dtype: object

Valores nulos
date            0
store_nbr       0
transactions    0
dtype: int64

Duplicados
0

Primeras filas


,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922


# TRANSACTIONS

1. ¿Qué representa una fila?

Cada fila de transactions.csv **representa el número total de transacciones realizadas en una tienda concreta durante un día determinado**.

Cada observación está definida por la combinación de:

- *date*: fecha de la observación.
- *store_nbr*: identificador de la tienda.
- *transactions*: número total de transacciones registradas ese día en dicha tienda.

A diferencia de train.csv, **este archivo no distingue entre familias de productos, sino que proporciona únicamente el número total de transacciones diarias por tienda**.

Por ejemplo, la primera fila indica que el 1 de enero de 2013 se registraron 770 transacciones en la tienda 25.

2. ¿Cómo se une?

El archivo transactions.csv se relaciona con train.csv mediante la combinación de las columnas:

- *date*
- *store_nbr*

Es decir, **para cada tienda y cada fecha se puede añadir al conjunto de entrenamiento el número total de transacciones registradas ese día**.

La **relación entre ambos conjuntos** es de tipo *uno a muchos (1:N)*:

En transactions.csv existe **un único registro por tienda y fecha**.
En train.csv existen **varios registros para esa misma tienda y fecha, uno por cada familia de productos**.

Al realizar la unión, el valor de transactions se asociará a todas las familias de productos correspondientes a esa tienda y fecha.

Aunque el número de transacciones puede ser una variable con gran capacidad predictiva, debe utilizarse con precaución. Si el objetivo es predecir las ventas de una fecha futura, el número de transacciones de ese mismo día no estaría disponible en el momento de realizar la predicción, por lo que su uso directo podría introducir fuga de información (data leakage). Este aspecto se analizará y justificará durante la fase de preparación de datos y modelado.

In [8]:
describe_dataframe(oil, "OIL")

OIL
Filas: 1,218
Columnas: 2

Tipos de datos
date           object
dcoilwtico    float64
dtype: object

Valores nulos
date           0
dcoilwtico    43
dtype: int64

Duplicados
0

Primeras filas


,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20


# OIL

1. ¿Tiene fechas?

Sí. El archivo oil.csv **contiene una columna denominada *date***, que **identifica la fecha asociada a cada registro del precio del petróleo**.

**Cada fila representa el valor del precio del petróleo correspondiente a un día determinado**.

Esta variable temporal permitirá relacionar el precio diario del petróleo con las ventas registradas en la misma fecha mediante una unión por la columna date.

2. ¿Tiene nulos?

Sí. El archivo presenta **43 valores nulos en la variable *dcoilwtico***, mientras que la **columna *date* no contiene valores ausentes**.

Estos valores nulos deberán analizarse y tratarse durante la fase de preparación de datos, evaluando distintas estrategias de imputación (por ejemplo, propagación del último valor conocido o interpolación temporal) antes de incorporarlos al conjunto de entrenamiento.

3. ¿Qué variable contiene?

El archivo contiene la variable *dcoilwtico*, que **representa el precio diario del *petróleo West Texas Intermediate (WTI)***.

Esta variable se considera un **factor externo** que podría influir indirectamente en las ventas, ya que el precio del petróleo puede afectar a la economía, los costes de transporte y el comportamiento del consumo.

Por ello, *dcoilwtico* se utilizará como una **variable predictora adicional** tras incorporarla al conjunto de entrenamiento mediante la fecha (date).


Este dataset pertenece a una cadena de supermercados de Ecuador. El precio del petróleo se incluye porque la economía ecuatoriana tiene una fuerte relación con este recurso, de modo que sus variaciones pueden influir, directa o indirectamente, en la actividad económica y en los patrones de consumo. No significa que exista necesariamente una relación fuerte con las ventas, sino que es una hipótesis que el análisis exploratorio y los modelos permitirán evaluar.

In [9]:
describe_dataframe(holidays, "HOLIDAYS")

HOLIDAYS
Filas: 350
Columnas: 6

Tipos de datos
date           object
type           object
locale         object
locale_name    object
description    object
transferred      bool
dtype: object

Valores nulos
date           0
type           0
locale         0
locale_name    0
description    0
transferred    0
dtype: int64

Duplicados
0

Primeras filas


,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False


# HOLIDAYS

1. ¿Qué tipos de festivos existen?

La **columna *type*** indica el **tipo de evento asociado a cada fecha**. En este dataset pueden encontrarse diferentes categorías, entre ellas:

- *Holiday*: festivo oficial.
- *Event*: evento especial (por ejemplo, acontecimientos relevantes).
- *Additional*: día festivo adicional.
- *Bridge*: puente entre un festivo y un fin de semana.
- *Transfer*: festivo trasladado a otra fecha.
- *Work Day*: día laborable establecido para compensar un festivo trasladado.

Estos eventos pueden influir significativamente en el comportamiento de las ventas, por lo que constituyen una fuente importante de información para el modelo predictivo.

2. ¿Qué significa locale?

La **columna *locale*** indica el **ámbito geográfico en el que el festivo o evento es aplicable**.

Puede tomar tres valores:

- *National*: afecta a todo el país.
- *Regional*: afecta únicamente a una provincia o región.
- *Local*: afecta únicamente a una ciudad concreta.

La **columna *locale_name*** especifica el **nombre de la región o ciudad correspondiente cuando el ámbito no es nacional**.

Esta información será útil para determinar qué tiendas se ven afectadas por cada festivo en función de su ubicación.

3. ¿Qué significa transferred?

La **columna *transferred*** es una **variable booleana (True / False)** que **indica si un festivo oficial ha sido trasladado a otra fecha**.

En algunos países, determinados festivos se cambian de día para favorecer fines de semana largos o facilitar la organización laboral.

- *False*: el festivo se celebra en la fecha indicada.
- *True*: el festivo ha sido trasladado y la fecha original ya no corresponde al día efectivo de celebración.

Este aspecto deberá tenerse en cuenta durante la preparación de los datos para representar correctamente el efecto de los festivos sobre las ventas.

En este dataset no basta con saber que un día es festivo: también es importante dónde se celebra ese festivo.

Por ejemplo:
- Un festivo *Local* en Quito solo debería afectar a las tiendas situadas en Quito.
- Un festivo *Regional* solo afectará a las tiendas de esa provincia.
- Un festivo *National* afectará a todas las tiendas del país.

Esto significa que, durante la fase de ingeniería de variables, no será suficiente con crear una variable is_holiday. Habrá que estudiar cómo incorporar correctamente la información geográfica de los festivos teniendo en cuenta la ubicación de cada tienda.

In [10]:
describe_dataframe(sample_submission, "SAMPLE SUBMISSION")

SAMPLE SUBMISSION
Filas: 28,512
Columnas: 2

Tipos de datos
id         int64
sales    float64
dtype: object

Valores nulos
id       0
sales    0
dtype: int64

Duplicados
0

Primeras filas


,id,sales
0,3000888,0.0
1,3000889,0.0
2,3000890,0.0
3,3000891,0.0
4,3000892,0.0


# SAMPLE_SUBMISSION

El archivo sample_submission.csv **proporciona un ejemplo del formato que debe tener el archivo de predicciones para ser enviado a Kaggle.**

No contiene información útil para el entrenamiento del modelo, sino **únicamente la estructura que debe respetarse al generar la solución final**.

5. ¿Qué formato espera Kaggle?

Kaggle espera un **archivo CSV con *dos columnas***:

*Columna* y **Descripción**
- *id*	**Identificador único de cada observación del conjunto de prueba**.
- *sales*	**Valor predicho por el modelo para las ventas correspondientes a cada observación**.

Por tanto, una vez entrenado el modelo, será necesario generar un archivo con exactamente este formato para evaluar el rendimiento de las predicciones en la competición.


# RANGO TEMPORAL

In [11]:
# Convertir las fechas a datetime

train["date"] = pd.to_datetime(train["date"])
test["date"] = pd.to_datetime(test["date"])
transactions["date"] = pd.to_datetime(transactions["date"])
oil["date"] = pd.to_datetime(oil["date"])
holidays["date"] = pd.to_datetime(holidays["date"])

In [12]:
datasets = {
    "train": train,
    "test": test,
    "transactions": transactions,
    "oil": oil,
    "holidays": holidays
}

for name, df in datasets.items():
    print(f"{name}:")
    print(f"Desde: {df['date'].min()}")
    print(f"Hasta: {df['date'].max()}")
    print()

train:
Desde: 2013-01-01 00:00:00
Hasta: 2017-08-15 00:00:00

test:
Desde: 2017-08-16 00:00:00
Hasta: 2017-08-31 00:00:00

transactions:
Desde: 2013-01-01 00:00:00
Hasta: 2017-08-15 00:00:00

oil:
Desde: 2013-01-01 00:00:00
Hasta: 2017-08-31 00:00:00

holidays:
Desde: 2012-03-02 00:00:00
Hasta: 2017-12-26 00:00:00



# CLAVES DE UNIÓN

In [13]:
for name, df in {
    "train": train,
    "test": test,
    "stores": stores,
    "transactions": transactions,
    "oil": oil,
    "holidays": holidays,
    "sample_submission": sample_submission
}.items():
    print(f"\n{name.upper()}")
    print(df.columns.tolist())


TRAIN
['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion']

TEST
['id', 'date', 'store_nbr', 'family', 'onpromotion']

STORES
['store_nbr', 'city', 'state', 'type', 'cluster']

TRANSACTIONS
['date', 'store_nbr', 'transactions']

OIL
['date', 'dcoilwtico']

HOLIDAYS
['date', 'type', 'locale', 'locale_name', 'description', 'transferred']

SAMPLE_SUBMISSION
['id', 'sales']


# DICCIONARIO DE VARIABLES

In [14]:
def data_dictionary(df, dataset_name):
    dictionary = pd.DataFrame({
        "Variable": df.columns,
        "Tipo": df.dtypes.astype(str).values,
        "Dataset": dataset_name
    })
    return dictionary

In [15]:
dict_train = data_dictionary(train, "train")
dict_test = data_dictionary(test, "test")
dict_stores = data_dictionary(stores, "stores")
dict_transactions = data_dictionary(transactions, "transactions")
dict_oil = data_dictionary(oil, "oil")
dict_holidays = data_dictionary(holidays, "holidays")

In [16]:
full_dictionary = pd.concat(
    [
        dict_train,
        dict_test,
        dict_stores,
        dict_transactions,
        dict_oil,
        dict_holidays,
    ],
    ignore_index=True,
)

display(full_dictionary)

,Variable,Tipo,Dataset
0,id,int64,train
1,date,datetime64[ns],train
2,store_nbr,int64,train
3,family,object,train
4,sales,float64,train
5,onpromotion,int64,train
6,id,int64,test
7,date,datetime64[ns],test
8,store_nbr,int64,test
9,family,object,test


# PREGUNTAS

### CARACTERIZACIÓN GLOBAL DEL DATASET

In [17]:
# =============================================================================
# CARACTERIZACIÓN GLOBAL DEL DATASET
# =============================================================================

# Convertir las fechas a datetime (si aún no se ha hecho)
train["date"] = pd.to_datetime(train["date"])
test["date"] = pd.to_datetime(test["date"])

print("=" * 80)
print("CARACTERIZACIÓN GLOBAL DEL DATASET")
print("=" * 80)

# =============================================================================
# 1. ¿Cuántas tiendas existen?
# =============================================================================

num_stores = train["store_nbr"].nunique()

print("\n1. ¿Cuántas tiendas existen?")
print(f"   Número de tiendas: {num_stores}")

CARACTERIZACIÓN GLOBAL DEL DATASET

1. ¿Cuántas tiendas existen?
   Número de tiendas: 54


In [18]:
# =============================================================================
# 2. ¿Cuántas familias de productos existen?
# =============================================================================

num_families = train["family"].nunique()

print("\n2. ¿Cuántas familias de productos existen?")
print(f"   Número de familias: {num_families}")

print("\nFamilias de productos:")
print(sorted(train["family"].unique()))


2. ¿Cuántas familias de productos existen?
   Número de familias: 33

Familias de productos:
['AUTOMOTIVE', 'BABY CARE', 'BEAUTY', 'BEVERAGES', 'BOOKS', 'BREAD/BAKERY', 'CELEBRATION', 'CLEANING', 'DAIRY', 'DELI', 'EGGS', 'FROZEN FOODS', 'GROCERY I', 'GROCERY II', 'HARDWARE', 'HOME AND KITCHEN I', 'HOME AND KITCHEN II', 'HOME APPLIANCES', 'HOME CARE', 'LADIESWEAR', 'LAWN AND GARDEN', 'LINGERIE', 'LIQUOR,WINE,BEER', 'MAGAZINES', 'MEATS', 'PERSONAL CARE', 'PET SUPPLIES', 'PLAYERS AND ELECTRONICS', 'POULTRY', 'PREPARED FOODS', 'PRODUCE', 'SCHOOL AND OFFICE SUPPLIES', 'SEAFOOD']


In [19]:
# =============================================================================
# 3. ¿Qué periodo cubren los datos?
# =============================================================================
print("\n3. ¿Qué periodo cubren los datos?")

print(f"   TRAIN: {train['date'].min().date()}  -->  {train['date'].max().date()}")
print(f"   TEST : {test['date'].min().date()}   -->  {test['date'].max().date()}")


3. ¿Qué periodo cubren los datos?
   TRAIN: 2013-01-01  -->  2017-08-15
   TEST : 2017-08-16   -->  2017-08-31


In [20]:
# =============================================================================
# 4. ¿Todos los días tienen observaciones?
# =============================================================================
full_range = pd.date_range(train["date"].min(), train["date"].max())

missing_dates = full_range.difference(train["date"].unique())

print("\n4. ¿Todos los días tienen observaciones?")

if len(missing_dates) == 0:
    print("   Sí. No existen fechas sin observaciones.")
else:
    print(f"   No. Existen {len(missing_dates)} fechas sin registros.")
    print(missing_dates)


4. ¿Todos los días tienen observaciones?
   No. Existen 4 fechas sin registros.
DatetimeIndex(['2013-12-25', '2014-12-25', '2015-12-25', '2016-12-25'], dtype='datetime64[ns]', freq=None)


In [21]:
# =============================================================================
# 5. ¿Hay una serie temporal por tienda y familia?
# =============================================================================
num_series = train.groupby(["store_nbr", "family"]).ngroups

print("\n5. ¿Hay una serie temporal por tienda y familia?")
print(f"   Número de series temporales: {num_series}")

print("\nPrimeras series:")
display(train.groupby(["store_nbr", "family"]).size().reset_index(name="n_registros").head())


5. ¿Hay una serie temporal por tienda y familia?
   Número de series temporales: 1782

Primeras series:


,store_nbr,family,n_registros
0,1,AUTOMOTIVE,1684
1,1,BABY CARE,1684
2,1,BEAUTY,1684
3,1,BEVERAGES,1684
4,1,BOOKS,1684


In [22]:
# =============================================================================
# 6. ¿La variable sales contiene ceros o valores extremos?
# =============================================================================
print("\n6. ¿La variable 'sales' contiene ceros o valores extremos?")

display(train["sales"].describe())

num_zeros = (train["sales"] == 0).sum()
pct_zeros = num_zeros / len(train) * 100

print(f"Ventas iguales a 0: {num_zeros:,}")
print(f"Porcentaje de ceros: {pct_zeros:.2f}%")

print(f"Valor mínimo: {train['sales'].min():.2f}")
print(f"Valor máximo: {train['sales'].max():.2f}")



6. ¿La variable 'sales' contiene ceros o valores extremos?


count    3.000888e+06
mean     3.577757e+02
std      1.101998e+03
min      0.000000e+00
25%      0.000000e+00
50%      1.100000e+01
75%      1.958473e+02
max      1.247170e+05
Name: sales, dtype: float64

Ventas iguales a 0: 939,130
Porcentaje de ceros: 31.30%
Valor mínimo: 0.00
Valor máximo: 124717.00


In [23]:
# =============================================================================
# 7. ¿Qué información está disponible en el momento de realizar la predicción?
# =============================================================================

print("\n7. Información disponible para la predicción")

print("""
Se dispone de:

- Fecha (date)
- Tienda (store_nbr)
- Familia de productos (family)
- Número de productos en promoción (onpromotion)
- Información de la tienda (stores.csv)
- Festivos y eventos (holidays_events.csv)
- Precio diario del petróleo (oil.csv)

Debe analizarse cuidadosamente el uso de la variable 'transactions',
ya que el número de transacciones del mismo día no estaría disponible
en un escenario real de predicción y podría provocar data leakage.
""")


7. Información disponible para la predicción

Se dispone de:

- Fecha (date)
- Tienda (store_nbr)
- Familia de productos (family)
- Número de productos en promoción (onpromotion)
- Información de la tienda (stores.csv)
- Festivos y eventos (holidays_events.csv)
- Precio diario del petróleo (oil.csv)

Debe analizarse cuidadosamente el uso de la variable 'transactions',
ya que el número de transacciones del mismo día no estaría disponible
en un escenario real de predicción y podría provocar data leakage.



# TABLA RESUMEN

In [24]:
# =============================================================================
# TABLA RESUMEN DEL CONJUNTO DE DATOS
# =============================================================================

# Asegurar que las fechas tienen formato datetime
train["date"] = pd.to_datetime(train["date"])
test["date"] = pd.to_datetime(test["date"])
transactions["date"] = pd.to_datetime(transactions["date"])
oil["date"] = pd.to_datetime(oil["date"])
holidays["date"] = pd.to_datetime(holidays["date"])

# Agrupar todos los datasets para calcular métricas globales
datasets = {
    "train": train,
    "test": test,
    "stores": stores,
    "transactions": transactions,
    "oil": oil,
    "holidays": holidays,
    "sample_submission": sample_submission
}

# Total de valores nulos en todos los archivos
total_missing_values = sum(
    df.isnull().sum().sum()
    for df in datasets.values()
)

# Total de registros duplicados en todos los archivos
total_duplicates = sum(
    df.duplicated().sum()
    for df in datasets.values()
)

# Número de fechas ausentes en train
full_date_range = pd.date_range(
    start=train["date"].min(),
    end=train["date"].max(),
    freq="D"
)

missing_train_dates = full_date_range.difference(
    train["date"].drop_duplicates()
)

# Número de series temporales tienda-familia
num_time_series = (
    train[["store_nbr", "family"]]
    .drop_duplicates()
    .shape[0]
)

# Número y porcentaje de ventas iguales a cero
num_zero_sales = (train["sales"] == 0).sum()
pct_zero_sales = num_zero_sales / len(train) * 100

# Tabla resumen
summary = pd.DataFrame({
    "Característica": [
        "Observaciones de entrenamiento",
        "Observaciones de prueba",
        "Número de tiendas",
        "Número de familias de productos",
        "Periodo del conjunto de entrenamiento",
        "Periodo del conjunto de prueba",
        "Duración del horizonte de predicción",
        "Variable objetivo",
        "Variables principales",
        "Variables externas",
        "Número de series temporales",
        "Fechas sin observaciones en train",
        "Ventas iguales a cero",
        "Porcentaje de ventas iguales a cero",
        "Valor mínimo de sales",
        "Valor máximo de sales",
        "Valores nulos totales",
        "Archivo con valores nulos",
        "Registros duplicados totales",
        "Frecuencia temporal",
        "Tipo de problema"
    ],
    "Valor": [
        f"{len(train):,}",
        f"{len(test):,}",
        train["store_nbr"].nunique(),
        train["family"].nunique(),
        (
            f"{train['date'].min().date()} "
            f"- {train['date'].max().date()}"
        ),
        (
            f"{test['date'].min().date()} "
            f"- {test['date'].max().date()}"
        ),
        f"{test['date'].nunique()} días",
        "sales",
        "date, store_nbr, family, onpromotion",
        "stores, transactions, oil, holidays_events",
        num_time_series,
        len(missing_train_dates),
        f"{num_zero_sales:,}",
        f"{pct_zero_sales:.2f}%",
        f"{train['sales'].min():,.2f}",
        f"{train['sales'].max():,.2f}",
        int(total_missing_values),
        (
            f"oil.csv: "
            f"{int(oil['dcoilwtico'].isnull().sum())} nulos"
        ),
        int(total_duplicates),
        "Diaria",
        "Predicción de múltiples series temporales"
    ]
})

display(summary)

,Característica,Valor
0,Observaciones de entrenamiento,"3,000,888"
1,Observaciones de prueba,"28,512"
2,Número de tiendas,54
3,Número de familias de productos,33
4,Periodo del conjunto de entrenamiento,2013-01-01 - 2017-08-15
5,Periodo del conjunto de prueba,2017-08-16 - 2017-08-31
6,Duración del horizonte de predicción,16 días
7,Variable objetivo,sales
8,Variables principales,"date, store_nbr, family, onpromotion"
9,Variables externas,"stores, transactions, oil, holidays_events"
